# TP3 - Clasificación de uso del suelo con CNNs (EuroSAT RGB)
### Tecnología Digital VI - Inteligencia Artificial

**Integrantes:** Manuel Milde, Luciano Pozzoli

Este notebook resuelve el TP3: clasificar el **uso del suelo** de recortes de
imágenes satelitales Sentinel-2 (dataset **EuroSAT**, versión **RGB**) en
**10 clases balanceadas** (`AnnualCrop`, `Forest`, `HerbaceousVegetation`,
`Highway`, `Industrial`, `Pasture`, `PermanentCrop`, `Residential`, `River`,
`SeaLake`) usando Redes Neuronales Convolucionales en **PyTorch**.

Estructura del notebook:

1. Setup y carga de datos
2. Análisis Exploratorio (EDA)
3. Pipeline de datos y Data Augmentation
4. Arquitectura custom (CNN desde cero)
5. Entrenamiento desde cero + monitoreo de métricas y de gradientes
6. Evaluación en test y análisis de errores (matriz de confusión)
7. Espacio latente (PCA / t-SNE / UMAP)
8. Transfer Learning con ResNet18 preentrenada en ImageNet
9. Comparación: CNN custom vs Transfer Learning
10. Conclusiones finales

La lógica reutilizable (datos, augmentation, modelos, loop de entrenamiento,
monitor de gradientes, evaluación y espacio latente) vive en
`src/eurosat_cnn/` — este notebook la importa y se enfoca en el análisis,
los resultados y las decisiones tomadas.

## 1. Setup y carga de datos

**Dónde correrlo:** se recomienda un entorno con GPU (Google Colab o un
notebook de Kaggle). En CPU funciona, pero el entrenamiento completo tarda
bastante más.

**Descarga del dataset** (https://www.kaggle.com/datasets/apollo2506/eurosat-dataset):

- **Kaggle notebook:** agregar el dataset con *Add Input* y setear
  `EUROSAT_DIR = "/kaggle/input/eurosat-dataset"`.
- **Colab / local:** con la API de Kaggle (necesita `kaggle.json`):
  ```bash
  pip install kaggle
  kaggle datasets download -d apollo2506/eurosat-dataset -p data/raw --unzip
  ```

El loader (`data.load_splits`) busca automáticamente la carpeta con las 10
subcarpetas de clase (ignora la versión multiespectral `EuroSATallBands`).
Si encuentra los CSVs `train.csv` / `validation.csv` / `test.csv` del dataset
de Kaggle usa esa partición; si no, hace un split estratificado 70/15/15
reproducible.

In [ ]:
# --- Solo en Colab: descomentar para clonar/subir el proyecto e instalar deps ---
# !pip install -q umap-learn kaggle
# from google.colab import files; files.upload()  # subir kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d apollo2506/eurosat-dataset -p data/raw --unzip

In [ ]:
import os
import sys
import time
from pathlib import Path

# Permite importar el paquete src/eurosat_cnn desde el notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Ruta al dataset (sobreescribir si está en otro lado, ej. en Kaggle)
# os.environ["EUROSAT_DIR"] = "/kaggle/input/eurosat-dataset"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn

from eurosat_cnn import config, data
from eurosat_cnn.utils import set_seed, get_device, count_params
from eurosat_cnn.models import EuroSATCNN, build_resnet18, resnet_param_groups
from eurosat_cnn.training import fit
from eurosat_cnn.gradients import GradientMonitor
from eurosat_cnn.evaluate import (
    predict, metrics_summary, report_df, plot_confusion_matrix,
    top_confusions, plot_misclassified,
)
from eurosat_cnn.latent import (
    extract_features, raw_pixels, project, separability_scores,
    plot_embeddings, plot_embedding_3d,
)
from eurosat_cnn.plotting import (
    plot_history, plot_comparison, epochs_to_reach,
    show_batch_by_class, show_augmentations,
)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = config.RANDOM_STATE
set_seed(RANDOM_STATE)
DEVICE = get_device()
print("Device:", DEVICE, "| PyTorch", torch.__version__)

In [ ]:
# --- Hiperparámetros del experimento ---------------------------------------
# SMOKE_TEST=True corre todo con muy pocos datos/épocas: sirve solo para
# verificar que el pipeline funciona de punta a punta (NO para reportar).
SMOKE_TEST = os.environ.get("EUROSAT_SMOKE") == "1"

EPOCHS_CUSTOM = 2 if SMOKE_TEST else 30   # CNN desde cero
EPOCHS_TL = 2 if SMOKE_TEST else 10       # ResNet18 (converge mucho antes)
GRAD_MONITOR_EPOCHS = 2 if SMOKE_TEST else 3
MAX_PER_CLASS = 30 if SMOKE_TEST else None
PRETRAINED = not SMOKE_TEST               # pesos de ImageNet (requiere internet)
TL_IMG_SIZE = 64                          # 224 = resolución de ImageNet (más lento)
BATCH_SIZE = 32 if SMOKE_TEST else config.BATCH_SIZE

def savefig(fig, name):
    fig.savefig(config.FIGURES_DIR / f"{name}.png", dpi=130, bbox_inches="tight")

In [ ]:
splits = data.load_splits(max_per_class=MAX_PER_CLASS)
splits["train"].head()

## 2. Análisis Exploratorio (EDA)

Objetivo: verificar el balance de clases y la integridad de los splits,
mirar ejemplos de cada clase para anticipar qué pares van a ser difíciles de
distinguir, y calcular las estadísticas por canal para normalizar.

### 2.1 Distribución de clases por split

In [ ]:
counts = pd.DataFrame({k: v["class_name"].value_counts() for k, v in splits.items()})
counts = counts.reindex(config.CLASSES)
counts["total"] = counts.sum(axis=1)
display(counts)

fig, ax = plt.subplots(figsize=(10, 4))
counts[["train", "val", "test"]].plot(kind="bar", ax=ax)
ax.set_title("Imágenes por clase y split")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); savefig(fig, "eda_clases"); plt.show()

# Chequeo de leakage: ninguna imagen debería estar en dos splits a la vez
paths = {k: set(v["path"]) for k, v in splits.items()}
print("Intersección train∩val:", len(paths["train"] & paths["val"]),
      "| train∩test:", len(paths["train"] & paths["test"]),
      "| val∩test:", len(paths["val"] & paths["test"]))

El dataset es **aproximadamente balanceado** (entre 2000 y 3000 imágenes por
clase en el EuroSAT completo; las clases con menos ejemplos son `Pasture`
—2000— e `Highway`/`Industrial`/`PermanentCrop`/`River` —2500—). Por eso
**accuracy** es una métrica razonable, pero igual reportamos **F1-macro** y
las métricas por clase para no esconder un mal desempeño en las clases más
chicas.

### 2.2 Ejemplos por clase y estadísticas por canal

In [ ]:
from torchvision import transforms as T

viz_ds = data.EuroSATDataset(splits["train"], T.ToTensor())
img0, _ = viz_ds[0]
print("Tamaño de imagen (C, H, W):", tuple(img0.shape))

fig = show_batch_by_class(viz_ds, splits["train"]["label"], (0, 0, 0), (1, 1, 1),
                          n_per_class=8, title="Ejemplos de train por clase")
savefig(fig, "eda_ejemplos"); plt.show()

In [ ]:
# Media/desvío por canal sobre train (para normalizar la CNN custom).
# Para ResNet18 se usan las estadísticas de ImageNet, que son las que vio
# la red durante su preentrenamiento.
EUROSAT_MEAN, EUROSAT_STD = data.compute_mean_std(splits["train"])
print("Media RGB:", np.round(EUROSAT_MEAN, 4))
print("Std   RGB:", np.round(EUROSAT_STD, 4))

In [ ]:
# Color medio por clase: ¿cuánta información trae solo el color?
rows = []
for c in config.CLASSES:
    sub = splits["train"][splits["train"]["class_name"] == c].sample(
        min(150, (splits["train"]["class_name"] == c).sum()), random_state=0)
    m = np.mean([viz_ds.transform(data.Image.open(p).convert("RGB")).mean((1, 2)).numpy()
                 for p in sub["path"]], axis=0)
    rows.append({"clase": c, "R": m[0], "G": m[1], "B": m[2]})
color_df = pd.DataFrame(rows).set_index("clase")

fig, ax = plt.subplots(figsize=(10, 4))
color_df.plot(kind="bar", color=["#d62728", "#2ca02c", "#1f77b4"], ax=ax)
ax.set_title("Intensidad media por canal y clase")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); savefig(fig, "eda_color_por_clase"); plt.show()

**Conclusiones del EDA** _(revisar/completar con lo observado)_:

- Clases como `SeaLake` (azul/oscuro y homogéneo) o `Forest` (verde oscuro,
  textura uniforme) son muy distintivas ya por color y textura → se espera
  que sean las más fáciles.
- `AnnualCrop`, `PermanentCrop`, `Pasture` y `HerbaceousVegetation` son todas
  coberturas vegetales con colores parecidos; se distinguen sobre todo por la
  **textura/geometría** (parcelas rectangulares en cultivos anuales, filas de
  plantas en cultivos permanentes, manchas irregulares en vegetación
  herbácea) → candidatas naturales a confundirse.
- `Highway` y `River` son estructuras **lineales** que cruzan el recorte con
  fondo vegetal; a 10 m/píxel una ruta y un río angosto pueden verse muy
  parecidos → otro par candidato a confusión.
- `Industrial` y `Residential` comparten superficies construidas; difieren en
  el tamaño de los edificios y la densidad.

## 3. Pipeline de datos y Data Augmentation

Las imágenes satelitales se toman en **vista cenital (nadir)**: no hay un
"arriba" o "abajo" canónico. Un campo de cultivo rotado 90° o espejado sigue
siendo exactamente la misma clase. Por eso las augmentations más pertinentes
son **geométricas e invariantes a la orientación** (implementadas en
`data.build_transforms`):

| Transformación | Justificación |
|---|---|
| `RandomHorizontalFlip` + `RandomVerticalFlip` | No hay orientación privilegiada en una imagen cenital. |
| `RandomRot90` (0/90/180/270°) | Igual que los flips; además, al ser múltiplos de 90° **no genera esquinas vacías** que haya que rellenar (a diferencia de una rotación arbitraria). Flips + rot90 cubren el grupo diédrico completo (8 orientaciones). |
| `RandomResizedCrop(scale=0.8-1)` | Robustez a pequeños desplazamientos y cambios de escala del recorte. Se mantiene suave porque la imagen es de solo 64×64. |
| `ColorJitter` suave (±10%) | Simula diferencias de iluminación, estación del año y atmósfera entre adquisiciones. Es **chico a propósito**: el color es muy informativo (agua vs. vegetación vs. suelo) y alterarlo mucho podría cambiar la semántica (ej. un pasto verde que "parezca" seco). |

**No** usamos augmentations que rompan la física de la imagen, como
rotaciones arbitrarias con relleno, *cutout* grandes o cambios de tono
(hue) fuertes. Validación y test **no** llevan augmentation (solo
normalización).

In [ ]:
train_tf, eval_tf = data.build_transforms(EUROSAT_MEAN, EUROSAT_STD, augment=True)
print("Train transform:\n", train_tf)
print("\nEval transform:\n", eval_tf)

loaders = data.make_dataloaders(splits, train_tf, eval_tf, batch_size=BATCH_SIZE)
xb, yb = next(iter(loaders["train"]))
print("\nBatch de train:", tuple(xb.shape), "| labels:", tuple(yb.shape))

In [ ]:
aug_ds = data.EuroSATDataset(splits["train"], train_tf)
for c in ["Highway", "AnnualCrop", "River"]:
    idx = int(np.where(splits["train"]["class_name"] == c)[0][0])
    fig = show_augmentations(aug_ds, idx, EUROSAT_MEAN, EUROSAT_STD, n=8,
                             title=f"8 augmentations de la misma imagen ({c})")
    savefig(fig, f"augment_{c}"); plt.show()

## 4. Arquitectura custom (CNN desde cero)

Diseñamos una CNN compacta estilo VGG (`models.EuroSATCNN`):

```
Input 3×64×64
 ├─ ConvBlock(3→32)    : [Conv3×3-BN-ReLU]×2 + MaxPool → 32×32×32
 ├─ ConvBlock(32→64)   : [Conv3×3-BN-ReLU]×2 + MaxPool → 64×16×16
 ├─ ConvBlock(64→128)  : [Conv3×3-BN-ReLU]×2 + MaxPool + Dropout2d → 128×8×8
 ├─ ConvBlock(128→256) : [Conv3×3-BN-ReLU]×2 + MaxPool + Dropout2d → 256×4×4
 ├─ Global Average Pooling → 256
 ├─ Dropout → Linear(256→128) → ReLU → Dropout     ← espacio latente (128-d)
 └─ Linear(128→10)                                  ← capa de clasificación
```

Decisiones de diseño:

- **Kernels 3×3 apilados**: dos convoluciones 3×3 tienen el mismo campo
  receptivo que una 5×5 con menos parámetros y una no-linealidad extra. Al
  final de la red el campo receptivo cubre todo el recorte de 64×64.
- **Duplicar canales al reducir la resolución a la mitad**: mantiene
  aproximadamente constante el costo por bloque y permite aprender cada vez
  más patrones (bordes → texturas → parcelas/estructuras).
- **Batch Normalization** después de cada convolución: estabiliza la
  distribución de las activaciones y la escala de los gradientes; es la
  principal defensa contra vanishing/exploding gradients (lo verificamos en
  la sección 5.2) y permite usar learning rates más altos.
- **Inicialización de He (Kaiming)**: diseñada para ReLU, preserva la
  varianza de activaciones y gradientes entre capas al inicio del
  entrenamiento.
- **Global Average Pooling** en lugar de `Flatten`: la clase de un recorte de
  EuroSAT depende de su textura global más que de la posición exacta de los
  objetos; GAP aporta esa invarianza y reduce mucho los parámetros de la
  parte densa (256×128 en vez de 4096×128).
- **Dropout** (en el bloque denso y `Dropout2d` en los últimos bloques
  convolucionales): regularización, el dataset (~19k imágenes de train) es
  chico para entrenar desde cero.

In [ ]:
set_seed(RANDOM_STATE)
custom_model = EuroSATCNN().to(DEVICE)
print(custom_model)
print("\nParámetros:", count_params(custom_model))

with torch.no_grad():
    out = custom_model(xb[:4].to(DEVICE))
    emb = custom_model.forward_features(xb[:4].to(DEVICE))
print("Logits:", tuple(out.shape), "| Embedding latente:", tuple(emb.shape))

## 5. Entrenamiento desde cero

- **Loss:** Cross-Entropy (clasificación multiclase).
- **Optimizador:** AdamW (`lr=1e-3`, `weight_decay=5e-4`): Adam con weight
  decay desacoplado, buena opción por defecto para CNNs desde cero.
- **Scheduler:** `OneCycleLR` (warm-up y luego decaimiento coseno, se
  actualiza en cada batch): acelera la convergencia y ayuda a terminar en un
  mínimo más "plano".
- **Selección del modelo:** guardamos los pesos de la época con mejor
  **accuracy de validación** (`fit` hace el *checkpointing* en memoria). El
  test solo se usa una vez, al final.
- **Monitoreo de gradientes:** `GradientMonitor` registra la norma L2 del
  gradiente de cada capa (conv/linear) cada 10 batches durante las primeras
  épocas.

### 5.1 Loop de entrenamiento

In [ ]:
set_seed(RANDOM_STATE)
custom_model = EuroSATCNN().to(DEVICE)
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=EPOCHS_CUSTOM,
    steps_per_epoch=len(loaders["train"]), pct_start=0.2,
)
grad_monitor = GradientMonitor(custom_model, max_epochs=GRAD_MONITOR_EPOCHS,
                               log_every=1 if SMOKE_TEST else 10)

hist_custom, _ = fit(
    custom_model, loaders, optimizer, DEVICE, epochs=EPOCHS_CUSTOM,
    scheduler=scheduler, grad_monitor=grad_monitor, name="CNN custom",
)
torch.save(custom_model.state_dict(), config.MODELS_DIR / "cnn_custom_best.pt")
hist_custom.to_csv(config.OUTPUTS_DIR / "history_cnn_custom.csv", index=False)

In [ ]:
fig = plot_history(hist_custom, "CNN custom (desde cero): evolución de Loss y Accuracy")
savefig(fig, "curvas_cnn_custom"); plt.show()
hist_custom.tail()

**Análisis de las curvas** _(completar con lo observado)_:

- Es esperable que en las primeras épocas la **accuracy de validación sea
  mayor que la de train** y la loss de validación menor: train se evalúa
  *con* augmentation y *con* dropout activo (más difícil), mientras que
  validación se evalúa con el modelo en modo `eval` y sobre imágenes sin
  transformar. No es un error.
- ¿Hay overfitting? Mirar si la loss de validación empieza a subir mientras
  la de train sigue bajando. Con augmentation + dropout + weight decay se
  espera una brecha chica.
- ¿En qué época se alcanza el mejor val accuracy? ¿Se estanca antes de
  terminar? (con `OneCycleLR` suele haber una mejora final cuando el LR
  decae).

### 5.2 Monitoreo del flujo de gradientes

Para cada capa con pesos (convoluciones y lineales, ordenadas de la entrada a
la salida) graficamos:

1. la norma del gradiente a lo largo de los batches de las primeras épocas,
2. la norma media por capa (con su rango min-max),
3. la norma global del gradiente.

**¿Qué buscamos?**

- **Vanishing gradients:** las primeras capas con normas **órdenes de
  magnitud** menores que las últimas (o tendiendo a 0) → no aprenden.
- **Exploding gradients:** normas que crecen sin control, picos enormes o
  `NaN`/`Inf` → la loss diverge.
- **Flujo sano:** normas del mismo orden de magnitud entre capas (diferencias
  de 1-2 órdenes son normales), estables o decreciendo suavemente a medida
  que la loss baja.

In [ ]:
fig = grad_monitor.plot("CNN custom: flujo de gradientes en las primeras épocas")
savefig(fig, "gradientes_cnn_custom"); plt.show()

display(grad_monitor.summary())
pd.Series(grad_monitor.diagnose(), name="diagnóstico")

**Contraste didáctico (opcional): qué pasaría sin BatchNorm ni ReLU.**
Para poner en contexto los números de arriba, entrenamos unos pocos batches
de una CNN **profunda (10 convs), sin BatchNorm y con activación sigmoide**
—la receta clásica del vanishing gradient: la derivada de la sigmoide es como
mucho 0.25, así que el gradiente se multiplica por factores < 1 en cada capa
al hacer backpropagation—. Si el diagnóstico es correcto, en esta red las
primeras capas deberían recibir gradientes varios órdenes de magnitud más
chicos que la última.

In [ ]:
RUN_GRADIENT_CONTRAST = True

if RUN_GRADIENT_CONTRAST:
    def plain_sigmoid_cnn(depth=10, ch=32):
        layers, in_ch = [], 3
        for i in range(depth):
            layers += [nn.Conv2d(in_ch, ch, 3, padding=1), nn.Sigmoid()]
            if i % 3 == 2:
                layers.append(nn.MaxPool2d(2))
            in_ch = ch
        return nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                             nn.Linear(ch, config.NUM_CLASSES))

    set_seed(RANDOM_STATE)
    bad_model = plain_sigmoid_cnn().to(DEVICE)
    bad_monitor = GradientMonitor(bad_model, max_epochs=1, log_every=1)
    bad_opt = torch.optim.SGD(bad_model.parameters(), lr=0.01)
    crit = nn.CrossEntropyLoss()
    bad_model.train()
    for step, (x, y) in enumerate(loaders["train"]):
        if step >= 30:
            break
        bad_opt.zero_grad()
        crit(bad_model(x.to(DEVICE)), y.to(DEVICE)).backward()
        bad_monitor.log(epoch=0)
        bad_opt.step()

    fig = bad_monitor.plot("Contraste: CNN profunda con sigmoide y sin BatchNorm")
    savefig(fig, "gradientes_contraste_sigmoide"); plt.show()

    cmp = pd.DataFrame({
        "CNN custom (BN+ReLU)": grad_monitor.summary()["grad_norm_mean"].values[[0, -1]],
        "CNN sigmoide sin BN": bad_monitor.summary()["grad_norm_mean"].values[[0, -1]],
    }, index=["primera capa", "última capa"]).T
    cmp["ratio última/primera"] = cmp["última capa"] / cmp["primera capa"]
    display(cmp.style.format("{:.2e}"))

**Conclusión sobre gradientes** _(completar con lo observado)_: en la CNN
custom, ¿las normas de la primera y la última capa están en el mismo orden de
magnitud? ¿Hay `NaN`/`Inf` o picos? Comparar el ratio última/primera capa con
el de la red con sigmoide sin BatchNorm, donde el gradiente se desvanece a
medida que retrocede hacia la entrada. Explicar el rol de BatchNorm, ReLU y
la inicialización de He en que la red custom **no** sufra este problema.

## 6. Evaluación en test y análisis de errores

Evaluamos **una única vez** el mejor modelo (según validación) sobre el
conjunto de test.

In [ ]:
y_true_c, y_pred_c, probs_c = predict(custom_model, loaders["test"], DEVICE)
test_custom = metrics_summary(y_true_c, y_pred_c)
print("CNN custom - test:", {k: round(v, 4) for k, v in test_custom.items()})
report_df(y_true_c, y_pred_c)

In [ ]:
fig, cm_custom = plot_confusion_matrix(y_true_c, y_pred_c, title="CNN custom - test")
savefig(fig, "confusion_cnn_custom"); plt.show()

conf_custom = top_confusions(cm_custom, k=8)
conf_custom

In [ ]:
test_ds_vis = data.EuroSATDataset(splits["test"], eval_tf)
pairs = list(conf_custom[["real", "predicho"]].head(4).itertuples(index=False, name=None))
fig = plot_misclassified(test_ds_vis, y_true_c, y_pred_c, probs_c,
                         EUROSAT_MEAN, EUROSAT_STD, pairs=pairs, n_per_pair=6)
if fig is not None:
    savefig(fig, "errores_cnn_custom"); plt.show()

**Análisis de las confusiones más frecuentes** _(validar contra la tabla y
las imágenes de arriba)_. Hipótesis a contrastar con los resultados:

- **`Highway` ↔ `River`**: ambas son estructuras **lineales** angostas que
  atraviesan el recorte sobre un fondo vegetal. A 10 m/píxel una autopista y
  un río angosto ocupan pocos píxeles de ancho y el color del agua (marrón
  verdoso, con sedimentos) puede parecerse al del asfalto o a la banquina.
- **`PermanentCrop` ↔ `AnnualCrop` / `HerbaceousVegetation`**: viñedos y
  frutales (cultivo permanente) muestran texturas en filas parecidas a las de
  los cultivos anuales, y cuando están poco desarrollados se confunden con
  vegetación herbácea. La diferencia es temporal/fenológica, información que
  una sola imagen RGB no tiene.
- **`Pasture` ↔ `HerbaceousVegetation`**: ambas son coberturas de pasto
  verde sin estructura geométrica marcada; la distinción es más de uso
  (ganadero) que visual.
- **`Industrial` ↔ `Residential`**: superficies construidas; se distinguen
  por el tamaño de los techos y la densidad de calles/vegetación.
- **Clases "fáciles"** (`Forest`, `SeaLake`): color y textura muy
  homogéneos y característicos → recall cercano a 1.

En general, los errores se concentran entre clases **semánticamente
vecinas** (todas vegetales, todas construidas, todas lineales) y no entre
clases dispares (ej. `SeaLake` vs `Industrial`), lo cual indica que la red
aprendió una representación con sentido. También pueden aparecer errores
por **ruido de etiquetado o imágenes mixtas** (ej. un recorte con un río y
una ruta a la vez), que ningún modelo puede resolver.

## 7. Espacio latente

Extraemos los features **justo antes de la capa de clasificación final**
(input de la última `nn.Linear`, capturado con un *forward hook*): para la
CNN custom es el embedding de **128 dimensiones**. Lo proyectamos a 2D con:

- **PCA**: proyección lineal; muestra la estructura global, pero no puede
  "desenrollar" variedades no lineales.
- **t-SNE**: preserva vecindarios locales (buena para ver clusters), pero las
  distancias entre clusters y su tamaño **no** son interpretables.
- **UMAP**: también no lineal, más rápido que t-SNE y suele preservar mejor
  la estructura global.

Como cuantificación complementaria calculamos el **silhouette score**
usando las clases reales como clusters (≈1: clusters compactos y separados;
≈0: solapados). Como **baseline** proyectamos también los **píxeles
crudos**: así vemos cuánto de la separabilidad la "construye" la red.

In [ ]:
feats_c, labels_c = extract_features(custom_model, loaders["test"], DEVICE)
print("Features latentes (test):", feats_c.shape)

t0 = time.perf_counter()
emb_custom = {
    "PCA": project(feats_c, "pca"),
    "t-SNE": project(feats_c, "tsne"),
    "UMAP": project(feats_c, "umap"),
}
print(f"Proyecciones calculadas en {time.perf_counter() - t0:.1f}s")

fig = plot_embeddings(emb_custom, labels_c, title="CNN custom: espacio latente (128-d) del set de test")
savefig(fig, "latente_cnn_custom"); plt.show()

In [ ]:
# Baseline: los mismos métodos sobre los píxeles crudos (3x64x64 = 12288-d)
X_raw, y_raw = raw_pixels(loaders["test"])
emb_raw = {"PCA (píxeles)": project(X_raw, "pca"), "t-SNE (píxeles)": project(X_raw, "tsne")}
fig = plot_embeddings(emb_raw, y_raw, title="Baseline: proyección de los píxeles crudos")
savefig(fig, "latente_pixeles"); plt.show()

sep = pd.concat([
    separability_scores(feats_c, labels_c, {f"{k} 2D (latente)": v for k, v in emb_custom.items()}),
    separability_scores(X_raw, y_raw, {"t-SNE 2D (píxeles)": emb_raw["t-SNE (píxeles)"]})
        .replace({"espacio": {f"latente ({X_raw.shape[1]}-d)": f"píxeles crudos ({X_raw.shape[1]}-d)"}}),
], ignore_index=True)
sep

In [ ]:
# Vista 3D (UMAP con 3 componentes)
emb3d = project(feats_c, "umap", n_components=3)
fig = plot_embedding_3d(emb3d, labels_c, title="CNN custom: UMAP 3D del espacio latente")
savefig(fig, "latente_cnn_custom_3d"); plt.show()

**Análisis del espacio latente** _(completar con lo observado)_:

- ¿Las clases forman **clusters distinguibles** antes de la capa final? Si la
  red clasifica bien, la capa final es solo un clasificador lineal sobre este
  embedding: buena separabilidad acá ⇔ buena accuracy.
- ¿Qué clusters están **pegados o se solapan**? Deberían coincidir con las
  confusiones de la matriz de la sección 6 (ej. `Highway`/`River`,
  `PermanentCrop`/`HerbaceousVegetation`/`AnnualCrop`, `Pasture`/
  `HerbaceousVegetation`).
- ¿Se observa la estructura **semántica** esperada (clases vegetales
  cercanas entre sí, construidas cerca entre sí, `SeaLake` aislado)?
  Recordar que en t-SNE la distancia *entre* clusters no es confiable; para
  eso mirar PCA/UMAP.
- Comparación con píxeles crudos: en el espacio de píxeles las clases están
  muy mezcladas (silhouette cercano a 0 o negativo), mientras que en el
  latente se separan → la red **aprendió** features discriminativos que no
  estaban linealmente disponibles en la entrada.

## 8. Transfer Learning: ResNet18 preentrenada en ImageNet

Elegimos **ResNet18**: es una arquitectura clásica, liviana (11M de
parámetros) y con *skip connections*, que garantizan un buen flujo de
gradientes aunque descongelemos toda la red.

**Adaptación a EuroSAT:**

- Reemplazamos la capa final `fc` (1000 clases de ImageNet) por una
  `Linear(512 → 10)` inicializada al azar.
- Normalizamos con la **media/desvío de ImageNet** (son las estadísticas con
  las que fueron entrenados los filtros) y usamos la misma augmentation que
  en la CNN custom.
- Resolución: `TL_IMG_SIZE = 64` (nativa de EuroSAT) para que la
  comparación de tiempo por época sea justa. Subir a 224 (resolución de
  ImageNet) suele mejorar algo la accuracy a costa de ~12× más cómputo.

**Dos estrategias** (`build_resnet18(trainable=...)`):

1. **Solo últimas capas** (`trainable="layer4"`): congelamos todo el
   backbone salvo el último bloque residual y la cabeza. Se entrenan menos
   parámetros y cada época es más barata (no hay que calcular gradientes de
   las primeras capas).
2. **Fine-tuning completo** (`trainable="all"`) con **learning rates
   discriminativos**: `1e-4` para el backbone (ajustar sin destruir lo
   aprendido en ImageNet) y `1e-3` para la cabeza nueva.

Hay un *domain shift* entre ImageNet (fotos de objetos tomadas de costado) y
EuroSAT (texturas cenitales), pero los filtros de las primeras capas (bordes,
colores, texturas) son genéricos y se transfieren bien.

In [ ]:
tl_train_tf, tl_eval_tf = data.build_transforms(
    config.IMAGENET_MEAN, config.IMAGENET_STD, img_size=TL_IMG_SIZE, augment=True
)
tl_loaders = data.make_dataloaders(splits, tl_train_tf, tl_eval_tf, batch_size=BATCH_SIZE)


def train_resnet(trainable, name, lr_backbone=1e-4, lr_head=1e-3, monitor=False):
    set_seed(RANDOM_STATE)
    model = build_resnet18(pretrained=PRETRAINED, trainable=trainable).to(DEVICE)
    print(f"{name} - parámetros:", count_params(model))
    groups = resnet_param_groups(model, lr_backbone=lr_backbone, lr_head=lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in groups], epochs=EPOCHS_TL,
        steps_per_epoch=len(tl_loaders["train"]), pct_start=0.2,
    )
    gm = GradientMonitor(model, max_epochs=GRAD_MONITOR_EPOCHS,
                         log_every=1 if SMOKE_TEST else 10) if monitor else None
    hist, _ = fit(model, tl_loaders, opt, DEVICE, epochs=EPOCHS_TL,
                  scheduler=sched, grad_monitor=gm, name=name)
    torch.save(model.state_dict(), config.MODELS_DIR / f"{name.replace(' ', '_')}.pt")
    hist.to_csv(config.OUTPUTS_DIR / f"history_{name.replace(' ', '_')}.csv", index=False)
    return model, hist, gm

### 8.1 ResNet18 descongelando solo `layer4` + `fc`

In [ ]:
resnet_l4, hist_l4, _ = train_resnet("layer4", "ResNet18 layer4+fc")
fig = plot_history(hist_l4, "ResNet18 (solo layer4 + fc): Loss y Accuracy")
savefig(fig, "curvas_resnet_layer4"); plt.show()

### 8.2 ResNet18 con fine-tuning completo

In [ ]:
resnet_ft, hist_ft, gm_ft = train_resnet("all", "ResNet18 fine-tuning", monitor=True)
fig = plot_history(hist_ft, "ResNet18 (fine-tuning completo): Loss y Accuracy")
savefig(fig, "curvas_resnet_finetuning"); plt.show()

In [ ]:
# Flujo de gradientes en el fine-tuning: las skip connections deberían
# mantener normas comparables entre las primeras y las últimas capas.
fig = gm_ft.plot("ResNet18 fine-tuning: flujo de gradientes en las primeras épocas")
savefig(fig, "gradientes_resnet_finetuning"); plt.show()
pd.Series(gm_ft.diagnose(), name="diagnóstico")

### 8.3 Evaluación en test de los modelos de transfer learning

In [ ]:
results_tl = {}
for name, model in [("ResNet18 layer4+fc", resnet_l4), ("ResNet18 fine-tuning", resnet_ft)]:
    yt, yp, pr = predict(model, tl_loaders["test"], DEVICE)
    results_tl[name] = {"y_true": yt, "y_pred": yp, "probs": pr, **metrics_summary(yt, yp)}
    print(name, {k: round(results_tl[name][k], 4) for k in ("accuracy", "f1_macro")})

best_tl_name = max(results_tl, key=lambda k: results_tl[k]["f1_macro"])
best_tl = results_tl[best_tl_name]
print("\nMejor variante de transfer learning:", best_tl_name)
report_df(best_tl["y_true"], best_tl["y_pred"])

In [ ]:
fig, cm_tl = plot_confusion_matrix(best_tl["y_true"], best_tl["y_pred"], title=f"{best_tl_name} - test")
savefig(fig, "confusion_resnet"); plt.show()

# ¿Qué confusiones se mantienen y cuáles resuelve el transfer learning?
conf_cmp = (
    top_confusions(cm_custom, k=10).rename(columns={"n_errores": "errores_custom"})
    .merge(top_confusions(cm_tl, k=45).rename(columns={"n_errores": f"errores_{best_tl_name}"})
           [["real", "predicho", f"errores_{best_tl_name}"]],
           on=["real", "predicho"], how="left")
    .fillna({f"errores_{best_tl_name}": 0})
    .drop(columns="pct_de_la_clase_real")
)
conf_cmp

In [ ]:
# Espacio latente de la mejor ResNet (512-d tras el average pooling)
best_tl_model = resnet_l4 if best_tl_name == "ResNet18 layer4+fc" else resnet_ft
feats_tl, labels_tl = extract_features(best_tl_model, tl_loaders["test"], DEVICE)
emb_tl = {"PCA": project(feats_tl, "pca"), "t-SNE": project(feats_tl, "tsne"),
          "UMAP": project(feats_tl, "umap")}
fig = plot_embeddings(emb_tl, labels_tl, title=f"{best_tl_name}: espacio latente (512-d) del set de test")
savefig(fig, "latente_resnet"); plt.show()

pd.concat([
    separability_scores(feats_c, labels_c, {"t-SNE 2D": emb_custom["t-SNE"]}).assign(modelo="CNN custom"),
    separability_scores(feats_tl, labels_tl, {"t-SNE 2D": emb_tl["t-SNE"]}).assign(modelo=best_tl_name),
], ignore_index=True)

## 9. Comparación: CNN custom vs Transfer Learning

Comparamos según lo que pide la consigna:

- **Velocidad de convergencia:** épocas (y minutos) necesarios para llegar a
  90% / 95% de accuracy de validación, y forma de las curvas.
- **Rendimiento final:** accuracy y F1-macro en test.
- **Tiempo de cómputo por época:** promedio del tiempo de entrenamiento por
  época (sin la primera, que incluye el *warm-up* de CUDA/DataLoader).

In [ ]:
histories = {"CNN custom": hist_custom, "ResNet18 layer4+fc": hist_l4,
             "ResNet18 fine-tuning": hist_ft}
fig = plot_comparison(histories)
savefig(fig, "comparacion_curvas"); plt.show()

In [ ]:
models_info = {"CNN custom": custom_model, "ResNet18 layer4+fc": resnet_l4,
               "ResNet18 fine-tuning": resnet_ft}
tests = {"CNN custom": test_custom, **{k: v for k, v in results_tl.items()}}

rows = []
for name, h in histories.items():
    p = count_params(models_info[name])
    t_ep = h["train_time_s"].iloc[1:] if len(h) > 1 else h["train_time_s"]
    rows.append({
        "modelo": name,
        "params_totales": p["total"],
        "params_entrenables": p["trainable"],
        "épocas": len(h),
        "val_acc_época_1": h["val_acc"].iloc[0],
        "épocas_hasta_90%": epochs_to_reach(h, 0.90),
        "épocas_hasta_95%": epochs_to_reach(h, 0.95),
        "mejor_val_acc": h["val_acc"].max(),
        "seg_por_época_train": t_ep.mean(),
        "min_totales": h["elapsed_s"].iloc[-1] / 60,
        "test_accuracy": tests[name]["accuracy"],
        "test_f1_macro": tests[name]["f1_macro"],
    })
comparison_df = pd.DataFrame(rows).set_index("modelo")
comparison_df.to_csv(config.OUTPUTS_DIR / "comparacion_modelos.csv")
comparison_df.T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
comparison_df["test_accuracy"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_ylim(max(0, comparison_df["test_accuracy"].min() - 0.05), 1)
axes[0].set_title("Accuracy en test")
comparison_df["seg_por_época_train"].plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("Segundos de entrenamiento por época")
(comparison_df["params_entrenables"] / 1e6).plot(kind="bar", ax=axes[2], color="#55A868")
axes[2].set_title("Parámetros entrenables (millones)")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); savefig(fig, "comparacion_barras"); plt.show()

**Discusión de la comparación** _(completar con los números de la tabla)_.
Puntos de partida:

- **Convergencia:** se espera que ResNet18 preentrenada supere el 90% de
  validación ya en la **primera o segunda época**, mientras que la CNN custom
  necesita varias épocas: parte de features genéricos ya aprendidos (bordes,
  texturas) en vez de pesos aleatorios.
- **Rendimiento final:** el fine-tuning completo suele quedar por encima
  (≈97-99% en EuroSAT RGB según la literatura) de la CNN custom (≈93-97% con
  augmentation). Descongelar solo `layer4` queda en el medio: las features
  intermedias de ImageNet no están adaptadas a texturas cenitales.
- **Cómputo por época:** ResNet18 tiene ~10× más parámetros que la CNN
  custom, así que cada época del fine-tuning completo es más cara. Congelar
  el backbone abarata la época (no se calculan gradientes ni se actualizan
  los pesos de las primeras capas), aunque el forward sigue siendo completo.
- **Trade-off:** aunque la época de ResNet sea más cara, necesita **muchas
  menos épocas**; el gráfico *accuracy vs minutos acumulados* muestra cuál
  llega antes a un nivel de accuracy dado. Si el presupuesto de cómputo o de
  tiempo de inferencia es muy limitado (ej. procesar teselas a escala
  continental), la CNN custom puede ser una alternativa razonable.

## 10. Conclusiones finales

_(completar)_: resumen de

- **Datos y augmentation:** por qué las invarianzas geométricas (flips +
  rotaciones de 90°) son las adecuadas para imágenes cenitales y su efecto en
  la brecha train/validación.
- **CNN custom:** arquitectura, accuracy/F1 en test, y la verificación de
  que no hubo vanishing/exploding gradients (y por qué: BatchNorm, ReLU, init
  de He).
- **Errores:** qué pares de clases se confunden y por qué (similitud
  visual/semántica, información temporal faltante, recortes mixtos).
- **Espacio latente:** las clases forman clusters separados antes de la
  capa final; los solapamientos coinciden con las confusiones.
- **Transfer learning:** cuánto mejora en accuracy y convergencia, a qué
  costo de cómputo por época, y cuál de los dos enfoques recomendarían según
  el caso de uso.